In [1]:
from huggingface_hub import HfApi, hf_hub_download
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split


In [ ]:
REPO_ID = "pyvene/axbench-concept10"
STORE = Path("../store/concept10")
STORE.mkdir(parents=True, exist_ok=True)
RAW_CSV = STORE / "raw.csv"
OUT_CSV = STORE / "wild.csv"


## Download & Raw CSV

In [3]:
# Download all Concept10 parquet subsets and keep only concept-containing positives.
# Map to concepts-compatible schema:
#   output_concept -> concept
#   model/layer/split path -> subtopic
#   input          -> question
#   output         -> answer
parquet_files = sorted(
    path
    for path in HfApi().list_repo_files(REPO_ID, repo_type="dataset")
    if path.endswith("/data.parquet")
)

raw_parts = []
for path in parquet_files:
    local_path = hf_hub_download(REPO_ID, path, repo_type="dataset")
    part = pd.read_parquet(local_path)
    part = part[part["category"] == "positive"].copy()
    part["subtopic"] = path.removesuffix("/data.parquet")

    raw_parts.append(pd.DataFrame({
        "concept":  part["output_concept"],
        "subtopic": part["subtopic"],
        "question": part["input"],
        "answer":   part["output"],
    }))

df_raw = pd.concat(raw_parts, ignore_index=True)

df_raw.to_csv(RAW_CSV, index=False)
print(df_raw.shape)
print(df_raw.groupby("concept").size().describe())


(4320, 4)
count     40.0
mean     108.0
std        0.0
min      108.0
25%      108.0
50%      108.0
75%      108.0
max      108.0
dtype: float64


## Cleaning

In [4]:
df = pd.read_csv(RAW_CSV)
df["concept"] = df["concept"].fillna("").str.strip()
df["subtopic"] = df["subtopic"].fillna("").str.strip()
df["question"] = df["question"].fillna("").str.strip()
df["answer"]   = df["answer"].fillna("").str.strip()

df = df[(df["concept"] != "") & (df["question"] != "") & (df["answer"] != "")]
df = df.drop_duplicates(subset=["concept", "question"]).reset_index(drop=True)

df.to_csv(OUT_CSV, index=False)
print(df.shape)
print(df.groupby("concept").size().sort_index())


(4319, 4)
concept
C/C++ programming syntax elements such as data types, function definitions, and variable declarations    108
HTML and XML tags or components related to documentation structure                                       108
biographical information about a person.                                                                 108
components related to user interface elements in programming                                             108
content related to code structure and definitions                                                        108
control flow statements and conditional logic in programming                                             108
functional aspects related to processes and operations in a system                                       107
instances of the word "onset" in relation to various conditions or phenomena                             108
layout attributes in a UI design context                                                                 108
m

## Subsets

In [5]:
# Balance by concept, then 80:20 train/test split
df = pd.read_csv(OUT_CSV).dropna(subset=["concept", "question", "answer"])

min_n = df.groupby("concept").size().min()
balanced = pd.concat(
    g.sample(min_n, random_state=42) for _, g in df.groupby("concept")
).reset_index(drop=True)

train_df, test_df = train_test_split(balanced, test_size=0.2, random_state=42)

train_df.to_csv(STORE / "train.csv", index=False)
test_df.to_csv(STORE / "test.csv", index=False)

print(f"min_n per concept: {min_n}")
print(f"balanced: {balanced.shape}, train: {train_df.shape}, test: {test_df.shape}")


min_n per concept: 107
balanced: (4280, 4), train: (3424, 4), test: (856, 4)


In [6]:
df_names = pd.read_csv(OUT_CSV)
names = sorted(df_names["concept"].dropna().astype(str).str.strip().unique())

print(list(names))

['C/C++ programming syntax elements such as data types, function definitions, and variable declarations', 'HTML and XML tags or components related to documentation structure', 'biographical information about a person.', 'components related to user interface elements in programming', 'content related to code structure and definitions', 'control flow statements and conditional logic in programming', 'functional aspects related to processes and operations in a system', 'instances of the word "onset" in relation to various conditions or phenomena', 'layout attributes in a UI design context', 'mathematical comparisons and relations', 'mathematical symbols and expressions', 'numeric expressions and dates', 'patterns or mentions related to numerical data and programming syntax in API contexts', 'phrases related to biological interactions and measurements', 'phrases that indicate magnitude or importance, particularly in scientific contexts', 'programming constructs and data structures in code 